# Polynomial Regression — Mathematics

**Goal.** Formalise polynomial regression as ordinary least squares on a transformed feature space. Define the polynomial **feature map** $\Phi$, write the model in matrix form, explain why the resulting design (the Vandermonde matrix) is numerically ill-conditioned, and introduce **orthogonal polynomial bases** that fix the conditioning without changing the model.

**Role of this notebook.** Pure mathematics — definitions, derivations, theorems. No code, no plots. Intuition is in `01_intuition.ipynb`; algorithms in `03_optimization.ipynb`; implementation in `05_hands_on_programming.ipynb`.

**Prerequisites.** `01_linear_regression/02_mathematics.ipynb` — the whole OLS apparatus (closed form, normal equations, hat matrix, gradient, Hessian) is reused verbatim. The only thing that changes here is the design matrix.

**Stage map.** `01_intuition` → **`02_mathematics`** → `03_optimization` → `04_statistics` → `05_hands_on_programming`.

**Five questions.**

1. What exactly is a "polynomial feature"? (The map $\Phi$.)
2. How does the OLS problem change when we apply $\Phi$?
3. What is the design matrix in the univariate case? (Vandermonde.)
4. How does polynomial regression generalise to $\mu$ltiple input variables?
5. Why does naive Vandermonde fail numerically, and what is the fix? (Orthogonal bases.)

---

**Reading conventions.** Same as `01_linear_regression/02_mathematics.ipynb`: definitions and theorems in blockquotes; $\mu$lti-line derivations in code blocks; equations numbered when later referenced.

## 0. Notation (additions to the linear-regression conventions)

We reuse everything from `01_linear_regression/02_mathematics.ipynb` §0 and add:

| Symbol | Type | Meaning |
|---|---|---|
| d | scalar $\in \mathbb{N}$ | polynomial **degree** (the maxi$\mu$m exponent allowed) |
| $\Phi$ | function $\mathbb{R}$ → $\mathbb{R}$^{d+1} (or $\mathbb{R}$^q → $\mathbb{R}$^p) | the **polynomial feature map** |
| $\Phi$(x) | vector | the feature vector after the map |
| $\Phi$(X) | matrix $\in \mathbb{R}^{n \times p}$ | the **polynomial design matrix**: row i is $\Phi$($x_i$)ᵀ |
| $V_d(x)$ | matrix $\in$ $\mathbb{R}^n$ˣ⁽ᵈ⁺¹⁾ | the **Vandermonde matrix** of degree d on n nodes |
| q | scalar $\in \mathbb{N}$ | number of original (raw) input variables |
| p | scalar $\in \mathbb{N}$ | number of features *after* $\Phi$ (for$\mu$la below) |

**Convention.** The single letter `x` denotes the original input — a *scalar* in §1–§3 (univariate setting) and a *vector* in §4 ($\mu$ltivariate). We always write $\Phi$(x) and $\Phi$(X) with the original input, so the same symbols cover both cases.

## 1. The polynomial feature map

### 1.1 Definition (univariate polynomial features)

Let x $\in$ $\mathbb{R}$. The **degree-d polynomial feature map** is the function

> $$\Phi_d : \mathbb{R} \to \mathbb{R}^{d+1}, \quad \Phi_d(x) := (1, x, x^2, x^3, \dots, x^d)^T$$   (1.1)

There are d + 1 entries; the leading 1 absorbs the intercept (the **bias trick** of `01_linear_regression/02_mathematics.ipynb` §1.3, made explicit by the feature map).

### 1.2 The polynomial model

The **degree-d polynomial regression model** is

> $$\hat{y}(x) := \langle \Phi_d(x), \theta \rangle = \theta_0 + \theta_1 x + \theta_2 x^2 + \dots + \theta_d x^d, \quad \theta \in \mathbb{R}^{d+1}$$   (1.2)

Equation (1.2) is *non-linear in x* — it is a degree-d polynomial in the scalar x. But it is **linear in the parameters $\theta^*$*: for fixed $\Phi_d(x)$, the map $\theta$ ↦ $\hat{y}$(x) is linear. This is the key observation that lets us reuse the entire linear-regression apparatus.

### 1.3 The polynomial design matrix

Given n observations (x_1, y_1), …, (xₙ, yₙ), stack the feature vectors as rows of a matrix:

> $$\Phi_d(X) := [ \Phi_d(x_1) \mid \Phi_d(x_2) \mid \dots \mid \Phi_d(x_n) ]^T \in \mathbb{R}^{n \times (d+1)}$$   (1.3)

Spelling it out entry by entry:

```
          ⎡ 1   x_1   x_1^2   x_1^3  …  x_1^d ⎤
          ⎢ 1   x_2   x_2^2   x_2^3  …  x_2^d ⎥
$\Phi_d(X)$ =  ⎢ ⋮   ⋮    ⋮     ⋮    ⋱    ⋮  ⎥                                      (1.4)
          ⎣ 1   xₙ   xₙ^2   xₙ^3  …  xₙ^d ⎦
```

Equation (1.4) is the **Vandermonde matrix** of degree d on nodes x_1, …, xₙ; we denote it $V_d(x)$ when the variable name needs to be explicit. The polynomial model in matrix form is then

> $$\hat{y} = \Phi_d(X) \theta \in \mathbb{R}^n$$                       (1.5)

which is **identical in shape** to the linear-regression model $\hat{y}$ = $X \theta$ from `01_linear_regression/02_mathematics.ipynb` (1.1) — only the design matrix has changed.

## 2. OLS on $\Phi$(X) — everything is inherited

Because the model (1.5) is linear in $\theta$, the entire OLS theory of the previous folder applies *verbatim*, with X replaced by $\Phi_d(X)$. We list the consequences without re-proving them; the proofs are word-for-word those in `01_linear_regression/02_mathematics.ipynb`.

### 2.1 Loss and closed form

The MSE loss is

> $$L(\theta) = \frac{1}{n} \|\Phi_d(X)\theta - y\|^2$$                                              (2.1)

with normal equations $\Phi_d(X)$ᵀ $\Phi_d(X)$ $\theta$ = $\Phi_d(X)$ᵀ y. When rank($\Phi_d(X)$) = d + 1 (see §3.2),

> $$\theta^* = (\Phi_d(X)^T \Phi_d(X))^{-1} \Phi_d(X)^T y$$   (2.2)

### 2.2 Gradient, Hessian, convexity

From Theorems 3.2–3.4 of `01_linear_regression/02_mathematics.ipynb`, applied to the polynomial design:

> $$\nabla L(\theta) = \frac{2}{n} \Phi_d(X)^T (\Phi_d(X)\theta - y)$$                                       (2.3)
>
> $\nabla^2 L(\theta)$ =  (2/n) $\cdot$ $\Phi_d(X)$ᵀ $\Phi_d(X)$   (constant, PSD; PD ⇔ rank = d+1).

L is convex; strictly convex iff the d + 1 columns of $\Phi_d(X)$ are linearly independent (§3.2 below shows this holds for *any* d + 1 distinct training nodes x_1, …, xₙ).

### 2.3 Geometry

The hat matrix H = $\Phi_d(X)$ ($\Phi_d(X)$ᵀ $\Phi_d(X)$)^{-1} $\Phi_d(X)$ᵀ is the orthogonal projection onto Col($\Phi_d(X)$) — the (d + 1)-dimensional subspace of $\mathbb{R}^n$ spanned by the d + 1 power columns. Predictions $\hat{y}$\* are still the closest point in that subspace to y.

**Reading.** Polynomial regression *is* OLS. Nothing about the loss, the gradient, or the geometry is new — only $\text{Col}(X)$ has grown from "all straight lines" (d = 1) to "all polynomials of degree $\le$ d" (general d).

## 3. The Vandermonde matrix

Two facts about $V_d(x)$ — one good (existence & uniqueness) and one bad (numerical conditioning) — together justify almost every algorithmic choice in `03_optimization.ipynb` and `05_hands_on_programming.ipynb`.

### 3.1 Definition recap

Repeating (1.4) for clarity: with nodes x = (x_1, …, xₙ) and degree d,

> $$V_d(x)$ := [ x_i^j ] \in \mathbb{R}^{n \times (d+1)}$$

### 3.2 Theorem (full column rank)

> **Theorem 3.2.** If at least d + 1 of the nodes x_1, …, xₙ are distinct, then rank($V_d(x)$) = d + 1, so $V_d(x)$ has full column rank.

**Proof sketch.** Equivalently, the only polynomial p(t) = $\sum$_j $\theta$_j tʲ of degree $\le$ d that vanishes on d + 1 distinct points is p ≡ 0 (fundamental theorem of algebra: a non-zero degree-d polynomial has at most d roots). Thus $V_d(x)$ $\theta$ = 0 forces $\theta$ = 0, i.e. $\text{Null}(V_d)$ = {0}. ∎

**Reading.** As long as we have at least d + 1 distinct x-values in the training set, the polynomial OLS problem has a *unique* solution. This is more lenient than it sounds — with n ≫ d the condition is overwhelmingly easy to satisfy.

### 3.3 Theorem (Vandermonde is ill-conditioned)

Existence and uniqueness say *something* about a problem; **conditioning** says how *easy* that problem is to solve numerically. Recall the condition number

> $$\kappa(A) := \frac{\sigma_{\max}(A)}{\sigma_{\min}(A)}$$

(see Theorem 6.1 of `01_linear_regression/02_mathematics.ipynb`). For Vandermonde matrices, $\kappa$ grows *exponentially* in the degree.

> **Theorem 3.3 (Gauts$\chi$ 1962, paraphrased).** For nodes x_1, …, x_{d+1} contained in a real interval of length L, the Vandermonde matrix $V_d(x)$ satisfies, for all sufficiently large d,
>
> $$\kappa($V_d(x)$) \ge c \cdot \rho^d$$    with some $\rho$ > 1 depending only on L.

*(We omit the full proof — it is a careful asymptotic analysis. The Gauts$\chi$ reference is the classic source: "On inverses of Vandermonde and confluent Vandermonde matrices", Numer. Math. 4 (1962).)*

**Practical consequence.** Squaring $\kappa$ when we form ($\Phi$ᵀ $\Phi$) (Theorem 6.1 of `01_linear_regression/02_mathematics.ipynb`) — together with the exponential growth in d — means the closed form (2.2) computed directly via the normal equations loses a few decimal digits per unit of degree. By d $\approx$ 15 on the interval [0, 1], double precision is exhausted: tiny noise in the inputs propagates into wildly different $\theta^*$.

## 4. Multivariate polynomial regression

When the original input is a *vector* x $\in$ $\mathbb{R}$ᵠ instead of a scalar, the polynomial map produces all monomials up to total degree d. Two equivalent definitions follow.

### 4.1 Definition ($\mu$ltivariate polynomial features)

For x = (x_1, …, x_q) $\in$ $\mathbb{R}$ᵠ, the **degree-d total polynomial feature map** is

> $$\Phi_{d,q}(x) := (x^{\alpha} : \alpha \in \mathbb{N}_0^q, \ |\alpha| \le d)$$    where  x^$\alpha$ := x_1^{$\alpha$_1} … x_q^{$\alpha$_q},  |$\alpha$| := $\alpha$_1 + … + $\alpha$_q.   (4.1)

**Example (q = 2, d = 2).** The monomials with |$\alpha$| $\le$ 2 are

> 1,    x_1,  x_2,    x_1^2,  x_1 x_2,  x_2^2.

So $\Phi$_{2, 2}(x) = (1, x_1, x_2, x_1^2, x_1 x_2, x_2^2)ᵀ $\in$ $\mathbb{R}$^6, and Col($\Phi$_{2,2}(X)) is the 6-dimensional space of bivariate polynomials of total degree $\le$ 2.

### 4.2 Counting features

The number p of features in $\Phi$_{d, q} is the number of $\mu$lti-indices $\alpha$ with |$\alpha$| $\le$ d in q variables:

> $$p = \binom{d+q}{q} = \frac{(d+q)!}{d! q!}$$   (4.2)

Some values:

| q \ d | 1 | 2 | 3 | 4 | 5 | 10 |
|---|---|---|---|---|---|---|
| 1 | 2 | 3 | 4 | 5 | 6 | 11 |
| 2 | 3 | 6 | 10 | 15 | 21 | 66 |
| 5 | 6 | 21 | 56 | 126 | 252 | 3 003 |
| 10 | 11 | 66 | 286 | 1 001 | 3 003 | 184 756 |

**Reading (curse of dimensionality).** With q = 10 original features and degree 5, the polynomial design has 3 003 columns. Each new feature steals one residual degree of freedom (recall the divisor n $-$ p in `01_linear_regression/04_statistics.ipynb` §6.2): unless n is huge, high-degree $\mu$ltivariate polynomial regression overfits almost immediately. This is one reason real ML practice tends to stay at d $\le$ 2–3 for vector inputs, and turn to non-parametric methods (kernels, trees, neural networks) when richer non-linearity is needed.

## 5. Orthogonal polynomial bases

Theorem 3.3 said the *monomial* Vandermonde basis is numerically nasty. The *mathematics* of the polynomial OLS problem does not depend on which basis we use — we can pick any d + 1 linearly independent polynomials. The natural fix is to pick a basis that is **orthogonal** with respect to a suitable inner product, so that $\Phi$ᵀ $\Phi$ becomes diagonal (or close to it).

### 5.1 Definition (orthogonal polynomial family)

Let w : [a, b] → $\mathbb{R}$₊ be a weight function. A family of polynomials {P_j}_{j=0}^$\infty$ with deg(P_j) = j is **orthogonal on [a, b] with weight w** if

> $$\langle P_j, P_k \rangle_w := \int_a^b P_j(t) P_k(t) w(t) dt = 0$$    whenever  j $\neq$ k.   (5.1)

Three classical families:

| Family | Interval | Weight w(t) | First few |
|---|---|---|---|
| Legendre | [$-$1, 1] | 1 | P_0=1, P_1=t, P_2=(3t^2 $-$ 1)/2 |
| Chebyshev (1st kind) | [$-$1, 1] | 1/$\sqrt$(1 $-$ t^2) | T_0=1, T_1=t, T_2=2t^2 $-$ 1 |
| Hermite (physicist's) | ($-$$\infty$, $\infty$) | e^{$-$t^2} | H_0=1, H_1=2t, H_2=4t^2 $-$ 2 |

### 5.2 Why this helps

Build the design matrix using an orthogonal family instead of monomials:

> Ψ_d(x) := ( P_0(x_1), P_1(x_1), …, P_d(x_1); … ; P_0(xₙ), …, P_d(xₙ) )  $\in$ $\mathbb{R}^n$ˣ⁽ᵈ⁺¹⁾.

When the training nodes $x_i$ are sampled appropriately (e.g. roots of P_{d+1} for Gauss quadrature, or uniformly on a grid) the **discrete** orthogonality

> $$\Psi_d(x)^T \Psi_d(x) \approx \text{diagonal}$$

holds approximately or exactly. Inverting a diagonal matrix is trivial: each coefficient $\theta$_j becomes a simple **inner product** between y and the j-th basis column, divided by $\|$P_j$\|$^2. Numerically, $\kappa$(Ψ_d) grows only **polynomially** in d for these families, not exponentially.

### 5.3 Equivalence to the monomial fit

Swit$\chi$ng basis does **not** change the fitted predictions $\hat{y}$\* = Hy — the column space of Ψ_d(X) is the same vector space as Col($\Phi_d(X)$) (both equal the span of all polynomials of degree $\le$ d). It is only the *coordinates* (the entries of $\hat{\theta}$) that change. The fitted polynomial p̂(x) = $\sum$_j $\hat{\theta}$_j P_j(x) is identical, term-for-term, to the monomial expression $\sum$_j $\tilde{\theta}$_j x^j after change of basis.

**Reading.** Numerical libraries that fit polynomial regression at non-trivial degrees (NumPy's `np.polynomial.polynomial.Polynomial.fit`, SciPy's `interpolate`, MATLAB's `polyfit` with the centring/scaling option, etc.) almost always work in an orthogonal basis internally and only convert back to monomial coefficients on request. This is the practical reason why "degree-20 polynomial fit" can succeed in modern software but failed in naive implementations 50 years ago.

## Takeaway

- **Feature map.**   $\Phi_d(x)$ := (1, x, x^2, …, x^d)ᵀ (eq. 1.1). The polynomial regression model $\hat{y}$ = $\Phi_d(X)$ $\theta$ (eq. 1.5) is non-linear in x but **linear in $\theta^*$*.
- **OLS inherits.**   Loss, gradient, Hessian, normal equations, hat matrix — all of `01_linear_regression/02_mathematics.ipynb` carries over verbatim with X replaced by $\Phi_d(X)$. The closed form is $\theta^*$ = ($\Phi_d(X)$ᵀ $\Phi_d(X)$)^{-1} $\Phi_d(X)$ᵀ y (eq. 2.2).
- **Existence (Theorem 3.2).**   d + 1 distinct training nodes ⇒ $V_d(x)$ has full column rank ⇒ $\theta^*$ is unique.
- **Conditioning (Theorem 3.3).**   $\kappa$(V_d) grows *exponentially* in d on a fixed interval. By d $\approx$ 15 in double precision the monomial basis is unusable.
- **Multivariate (eq. 4.2).**   The total-degree feature map produces C(d + q, q) features — explodes fast in q (curse of dimensionality). High-degree $\mu$ltivariate polynomials are rarely the right tool.
- **Orthogonal bases (§5).**   Legendre / Chebyshev / Hermite polynomials make $\Phi$ᵀ $\Phi$ approximately diagonal, with $\kappa$ growing only polynomially. The fitted *predictions* are unchanged; only the coordinates $\hat{\theta}$ change. This is what `np.polynomial.Polynomial.fit` does internally.

Next: `03_optimization.ipynb` — same gradient-descent algorithm as `01_linear_regression/03_optimization.ipynb`, but now we see in practice how the Vandermonde conditioning of Theorem 3.3 slows GD to a crawl, and how feature scaling / orthogonal bases bring it back.